# again new start


In [213]:
import pandas as pd
import string
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [214]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
df = pd.read_csv("../data/student_support_dataset.csv")

df.head()

,ID,Category,Question,Answer
0,1,Admissions,How can I apply for admission?,You can apply through the institution's online...
1,2,Admissions,What documents are required for admission?,"Academic transcripts, identity proof, photogra..."
2,3,Admissions,What are the eligibility criteria for admission?,Eligibility depends on the program.
3,4,Admissions,Is there an application fee?,Application fee requirements vary.
4,5,Admissions,When does the admission process begin?,Admission dates are announced on the official ...


In [219]:
stop_words = set(stopwords.words("english"))

In [220]:
def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Split into words
    words = word_tokenize(text)


    # Join back into sentence
    return " ".join(words)

In [221]:
sentence = "Where is the Library???"

print(preprocess_text(sentence))

where is the library


In [222]:
sentence = "How can I pay my tuition fees online?"

print(preprocess_text(sentence))

how can i pay my tuition fees online


In [223]:
df["Processed_Question"] = df["Question"].apply(preprocess_text)

In [224]:
df.head()

,ID,Category,Question,Answer,Processed_Question
0,1,Admissions,How can I apply for admission?,You can apply through the institution's online...,how can i apply for admission
1,2,Admissions,What documents are required for admission?,"Academic transcripts, identity proof, photogra...",what documents are required for admission
2,3,Admissions,What are the eligibility criteria for admission?,Eligibility depends on the program.,what are the eligibility criteria for admission
3,4,Admissions,Is there an application fee?,Application fee requirements vary.,is there an application fee
4,5,Admissions,When does the admission process begin?,Admission dates are announced on the official ...,when does the admission process begin


In [225]:
vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1,3),      # Use 1-word, 2-word, and 3-word phrases
    sublinear_tf=True,
    norm='l2'
)

In [226]:
question_vectors = vectorizer.fit_transform(df["Processed_Question"])


In [227]:
question_vectors.shape

(96, 684)

In [228]:
def get_response(user_question):

    # Clean user input
    processed_question = preprocess_text(user_question)
    print("Processed Question:", processed_question)

    # Convert to TF-IDF vector
    user_vector = vectorizer.transform([processed_question])

    # Calculate similarity
    similarity = cosine_similarity(user_vector, question_vectors)

    # Find best matching question
    best_match = similarity.argmax()

    # Highest similarity score
    best_score = similarity[0][best_match]

    # If similarity is too low
    if best_score < 0.60:
        return {
            "answer": "Sorry, I don't have information about that.",
            "category": None,
            "matched_question": None,
            "score": best_score
        }

    # Return matching information
    return {
        "answer": df.iloc[best_match]["Answer"],
        "category": df.iloc[best_match]["Category"],
        "matched_question": df.iloc[best_match]["Question"],
        "score": best_score
    }

In [229]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [230]:
import pandas as pd
import string
import re

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [231]:
lemmatizer = WordNetLemmatizer()

In [232]:
synonyms = {

    "college": "institution",
    "university": "institution",
    "campus": "institution",

    "admission": "admission",
    "admissions": "admission",
    "enroll": "admission",
    "enrollment": "admission",
    "apply": "admission",
    "joining": "admission",

    "fees": "fee",
    "payment": "fee",
    "payments": "fee",

    "hostels": "hostel",

    "books": "book",
    "libraries": "library",

    "exams": "exam",
    "examinations": "exam",

    "marks": "result",
    "grades": "result"
}

In [233]:
def preprocess_text(text):

    # lowercase
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    words = word_tokenize(text)

    processed = []

    for word in words:

        # synonym normalization
        if word in synonyms:
            word = synonyms[word]

        # lemmatization
        word = lemmatizer.lemmatize(word)

        processed.append(word)

    return " ".join(processed)

In [234]:
while True:

    question = input("\nAsk your question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    result = get_response(question)

    print("\nCategory:", result["category"])
    print("Matched Question:", result["matched_question"])
    print("Similarity Score:", round(result["score"], 2))
    print("\nChatbot:", result["answer"])

Processed Question: what is the hostel fee

Category: Hostel
Matched Question: how much is the hostel fee
Similarity Score: 0.61

Chatbot: you can find the option to pay the fees at the official website of the college
Processed Question: how can i pay the fee

Category: None
Matched Question: None
Similarity Score: 0.41

Chatbot: Sorry, I don't have information about that.
Goodbye!
